<a href="https://colab.research.google.com/github/TonyKaito/monad-test1/blob/main/Monad_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Implementation

In [1]:
class WriterMonad:
  def __init__(self, value, log=[]):
    self.value = value
    self.log = log

  def bind(self, funct):
    result = funct(self.value)
    return WriterMonad(result.value, self.log + [result.log])

  # Unit Test Purposes
  def __eq__(self, other):
    if not isinstance(other, WriterMonad):
        return NotImplemented
    return self.value == other.value and self.log == other.log

##### Initial Playgrounding

In [2]:
dict_test = {'t' : 124, 'w' : 234, 'test': 224}

In [3]:
result = WriterMonad("test") \
  .bind(lambda x : WriterMonad(x, x[0])) \
  .bind(lambda x : WriterMonad(x, dict_test[x]))

In [4]:
result.log

['t', 224]

### Test

In [5]:
import unittest

In [6]:
class MonadUnitTest(unittest.TestCase):
  def testMonad(self):
    dict_test = {'t' : 124, 'w' : 234, 'test': 224}
    result = WriterMonad("test") \
      .bind(lambda x : WriterMonad(x, x[0])) \
      .bind(lambda x : WriterMonad(x, dict_test[x]))

    self.assertEqual(result.value, 'test')
    self.assertEqual(result.log, ['t', 224])

In [7]:
unittest.main(argv=['first-arg-is-ignored'], exit=False)

.
----------------------------------------------------------------------
Ran 1 test in 0.003s

OK


### Some Game Logic Stuff

In [8]:
from dataclasses import dataclass

@dataclass
class gameData:
  id: list
  health: list
  debuff: list

In [9]:
def applyDamage(game_data_instance, player_id, damage):
  player_index = game_data_instance.id.index(player_id)
  game_data_instance.health[player_index] -= damage # Modify the list in place
  return game_data_instance # Return the modified instance

In [10]:
sample_ids = [1, 2, 3, 4, 5]
sample_health = [100, 85, 120, 90, 75]
sample_debuff = ['none', 'poison', 'slow', 'none', 'burn']

my_game_data = gameData(id=sample_ids, health=sample_health, debuff=sample_debuff)
print(my_game_data)

gameData(id=[1, 2, 3, 4, 5], health=[100, 85, 120, 90, 75], debuff=['none', 'poison', 'slow', 'none', 'burn'])


In [11]:
# Apply damage imperatively to player 3
applyDamage(my_game_data, 3, 20)

print("Game data after imperative damage to player 3:")
print(my_game_data)

Game data after imperative damage to player 3:
gameData(id=[1, 2, 3, 4, 5], health=[100, 85, 100, 90, 75], debuff=['none', 'poison', 'slow', 'none', 'burn'])


### Trying Monad Stuff

In [12]:
import copy # handling pass by reference issues for logging

def writerApplyDamage(game_data_instance, player_id, damage):
  applyDamage(game_data_instance, player_id, damage)
  return WriterMonad(player_id, copy.deepcopy(game_data_instance))

In [13]:
sample_ids = [1, 2, 3, 4, 5]
sample_health = [100, 85, 120, 90, 75]
sample_debuff = ['none', 'poison', 'slow', 'none', 'burn']

my_game_data = gameData(id=sample_ids, health=sample_health, debuff=sample_debuff)
print(my_game_data)

gameData(id=[1, 2, 3, 4, 5], health=[100, 85, 120, 90, 75], debuff=['none', 'poison', 'slow', 'none', 'burn'])


In [14]:
result = WriterMonad(3) \
  .bind(lambda x : writerApplyDamage(my_game_data, x, 20))

In [15]:
result.log

[gameData(id=[1, 2, 3, 4, 5], health=[100, 85, 100, 90, 75], debuff=['none', 'poison', 'slow', 'none', 'burn'])]

In [16]:
result = WriterMonad(3) \
  .bind(lambda x : writerApplyDamage(my_game_data, x, 20)) \
  .bind(lambda x : writerApplyDamage(my_game_data, x, 20))

In [17]:
result.log

[gameData(id=[1, 2, 3, 4, 5], health=[100, 85, 80, 90, 75], debuff=['none', 'poison', 'slow', 'none', 'burn']),
 gameData(id=[1, 2, 3, 4, 5], health=[100, 85, 60, 90, 75], debuff=['none', 'poison', 'slow', 'none', 'burn'])]

### Test

In [18]:
import copy

class MonadUnitTest2(unittest.TestCase):
  def testMonadGame(self):
    sample_ids = [1, 2, 3, 4, 5]
    sample_health = [100, 85, 120, 90, 75]
    sample_debuff = ['none', 'poison', 'slow', 'none', 'burn']

    my_game_data = gameData(id=sample_ids, health=sample_health, debuff=sample_debuff)
    id = 3

    result = WriterMonad(id) \
      .bind(lambda x : writerApplyDamage(my_game_data, x, 20)) \
      .bind(lambda x : writerApplyDamage(my_game_data, x, 20))

    should_be = [gameData(id=[1, 2, 3, 4, 5], health=[100, 85, 100, 90, 75], debuff=['none', 'poison', 'slow', 'none', 'burn']),
 gameData(id=[1, 2, 3, 4, 5], health=[100, 85, 80, 90, 75], debuff=['none', 'poison', 'slow', 'none', 'burn'])]

    self.assertEqual(result.value, id)
    self.assertEqual(result.log, should_be)

In [19]:
unittest.main(argv=['first-arg-is-ignored'], exit=False)

..
----------------------------------------------------------------------
Ran 2 tests in 0.002s

OK


AI Stuff Temp reference

In [20]:
# # --- Temporary monadic definitions for demonstration within this cell ---
# from dataclasses import dataclass

# # A monadic WriterMonad implementation
# class MonadicWriterMonad:
#   def __init__(self, value, log=None):
#     self.value = value
#     # Ensure log is a new list if default mutable argument is used
#     self.log = [] if log is None else list(log)

#   def bind(self, funct): # For a Monad, funct must be of type: a -> M b
#     # Apply the function to the current value, expecting it to return a MonadicWriterMonad
#     new_monad_result = funct(self.value)
#     # Combine the logs and return a new MonadicWriterMonad with the new value
#     return MonadicWriterMonad(new_monad_result.value, self.log + new_monad_result.log)

#   def __eq__(self, other):
#     if not isinstance(other, MonadicWriterMonad):
#         return NotImplemented
#     return self.value == other.value and self.log == other.log

#   def __repr__(self, ):
#       return f"MonadicWriterMonad(value={self.value!r}, log={self.log!r})"

# # A factory function to create a monadic applyDamage function.
# # This partial application allows `monadic_applyDamage` to receive `game_data_instance` and `damage_amount`,
# # and then return a function that expects only the `player_id` from the monad's value.
# def monadic_applyDamage_factory(game_data_instance_ref, damage_amount):
#   # This inner function closes over game_data_instance_ref and damage_amount
#   def _inner_monadic_applyDamage(player_id_val):
#     # Find player index and apply damage (modifying the game_data_instance_ref in place)
#     player_index = game_data_instance_ref.id.index(player_id_val)
#     original_health = game_data_instance_ref.health[player_index]
#     game_data_instance_ref.health[player_index] -= damage_amount
#     updated_health = game_data_instance_ref.health[player_index]

#     log_message = f"Applied {damage_amount} damage to player {player_id_val}. Health: {original_health} -> {updated_health}"
#     # Crucially, this returns a MonadicWriterMonad, passing the player_id along for subsequent operations
#     return MonadicWriterMonad(player_id_val, [log_message])
#   return _inner_monadic_applyDamage

# # --- End of temporary monadic definitions ---

# # Reset my_game_data to ensure a clean state for this demonstration within this cell
# # (This ensures the demonstration is self-contained and reproducible)
# sample_ids = [1, 2, 3, 4, 5]
# sample_health = [100, 85, 120, 90, 75]
# sample_debuff = ['none', 'poison', 'slow', 'none', 'burn']
# my_game_data = gameData(id=sample_ids, health=sample_health, debuff=sample_debuff)

# print("Initial my_game_data for monadic demonstration:", my_game_data)

# # Create the monadic damage application function
# damage_20_to_player = monadic_applyDamage_factory(my_game_data, 20)

# # Use the MonadicWriterMonad with the monadic damage function
# result_monadic = MonadicWriterMonad(3) \
#   .bind(damage_20_to_player) \
#   .bind(damage_20_to_player)

# print("\nResult of monadic chain (player 3 took 2x20 damage):")
# print(f"Value: {result_monadic.value}") # Still holds the player_id (3)
# print(f"Log: {result_monadic.log}") # Accumulated logs
# print(f"Final my_game_data state:", my_game_data) # Global game data is updated twice